## Python - Data Uploading and Data Inspection

In [2]:
# Importing packages
from pathlib import Path
import pandas as pd

In [3]:
# Defining root where I store the raw dataset
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Online Retail.xlsx"
)
staging_folder = Path("../data/staging")


print(RAW_DATA_PATH)
print("File exists:", RAW_DATA_PATH.exists())

staging_folder.mkdir(parents=True, exist_ok=True)

C:\Users\hdat2\Online-Retail-Analytics\data\raw\Online Retail.xlsx
File exists: True


In [4]:
# Loading the dataset
df = pd.read_excel(
    RAW_DATA_PATH,
    engine="openpyxl"
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [5]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [6]:
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 541,909
Number of columns: 8


In [7]:
data_types = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values
})

data_types

,column,data_type
0,InvoiceNo,object
1,StockCode,object
2,Description,object
3,Quantity,int64
4,InvoiceDate,datetime64[ns]
5,UnitPrice,float64
6,CustomerID,float64
7,Country,object


In [23]:
df.columns = df.columns.str.lower()
df["customerid"] = df["customerid"].astype("Int64")
df[["customerid"]].head()

,customerid
0,17850
1,17850
2,17850
3,17850
4,17850


In [25]:
# Standardise column names - for the use of SQL syntax

df = df.rename(columns={
    "invoiceno": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "unitprice": "unit_price",
    "customerid": "customer_id",
})

df.columns.tolist()

['invoice_no',
 'stock_code',
 'description',
 'quantity',
 'invoice_date',
 'unit_price',
 'customer_id',
 'country']

In [27]:
# Check missing values
missing_summary = (
    df.isna()
    .sum()
    .to_frame(name="missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

missing_summary.sort_values(
    by="missing_count",
    ascending=False
)

,missing_count,missing_percentage
customer_id,135080,24.93
description,1454,0.27
invoice_no,0,0.00
stock_code,0,0.00
quantity,0,0.00
invoice_date,0,0.00
unit_price,0,0.00
country,0,0.00


In [29]:
# Check duplicates
duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_count / len(df) * 100:.2f}%")

duplicate_rows = df[
    df.duplicated(keep=False)
].sort_values(
    by=["invoice_no", "stock_code", "invoice_date"]
)

duplicate_rows.head(20)

Exact duplicate rows: 5,268
Duplicate percentage: 0.97%


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom


In [31]:
# Any unusual figures? 
numeric_quality_summary = pd.Series({
    "negative_quantity": (df["quantity"] < 0).sum(),
    "zero_quantity": (df["quantity"] == 0).sum(),
    "negative_unit_price": (df["unit_price"] < 0).sum(),
    "zero_unit_price": (df["unit_price"] == 0).sum()
}).to_frame(name="row_count")

numeric_quality_summary

,row_count
negative_quantity,10624
zero_quantity,0
negative_unit_price,2
zero_unit_price,2515


In [33]:
# Check cancellations

cancelled_mask = (
    df["invoice_no"]
    .astype(str)
    .str.startswith("C", na=False)
)

cancelled_summary = pd.Series({
    "cancelled_rows": cancelled_mask.sum(),
    "cancelled_invoices": df.loc[
        cancelled_mask,
        "invoice_no"
    ].nunique()
})
# The cancelled rows are the total of invoices starting with "C" whereas the cancelled_invoices are the distinct count of those total invoices.
cancelled_summary
        

cancelled_rows        9288
cancelled_invoices    3836
dtype: int64

In [35]:
# Check timeframe

print("Earliest transaction:", df["invoice_date"].min())
print("Latest transaction:", df["invoice_date"].max())

Earliest transaction: 2010-12-01 08:26:00
Latest transaction: 2011-12-09 12:50:00


In [37]:
# Check unique values

unique_summary = pd.Series({
    "unique_invoices": df["invoice_no"].nunique(),
    "unique_products": df["stock_code"].nunique(),
    "unique_customers": df["customer_id"].nunique(),
    "unique_countries": df["country"].nunique()
}).to_frame(name="unique_count")

unique_summary

,unique_count
unique_invoices,25900
unique_products,4070
unique_customers,4372
unique_countries,38


In [39]:
# Summary table

quality_report = pd.DataFrame({
    "metric": [
        "Total rows",
        "Total columns",
        "Exact duplicate rows",
        "Missing customer IDs",
        "Missing descriptions",
        "Negative quantities",
        "Zero quantities",
        "Negative unit prices",
        "Zero unit prices",
        "Cancelled transaction rows",
        "Unique invoices",
        "Unique products",
        "Unique customers",
        "Unique countries"
    ],
    "value": [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df["customer_id"].isna().sum(),
        df["description"].isna().sum(),
        (df["quantity"] < 0).sum(),
        (df["quantity"] == 0).sum(),
        (df["unit_price"] < 0).sum(),
        (df["unit_price"] == 0).sum(),
        cancelled_mask.sum(),
        df["invoice_no"].nunique(),
        df["stock_code"].nunique(),
        df["customer_id"].nunique(),
        df["country"].nunique()
    ]
})

quality_report

,metric,value
0,Total rows,541909
1,Total columns,8
2,Exact duplicate rows,5268
3,Missing customer IDs,135080
4,Missing descriptions,1454
5,Negative quantities,10624
6,Zero quantities,0
7,Negative unit prices,2
8,Zero unit prices,2515
9,Cancelled transaction rows,9288


In [43]:
# Exporting the quality report

quality_report.to_csv(
    staging_folder / "data_quality_summary.csv",
    index=False
)

In [45]:
staging_df = df.copy()

output_file = staging_folder / "online_retail_staging.csv"

staging_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d %H:%M:%S",
    na_rep=""
)

print("CSV exported to:", output_file.resolve())

CSV exported to: C:\Users\hdat2\Online-Retail-Analytics\data\staging\online_retail_staging.csv


In [46]:
# Final check

validation_df = pd.read_csv(
    output_file,
    low_memory=False
)

print("Original rows:", len(staging_df))
print("Exported CSV rows:", len(validation_df))
print("Same row count:", len(staging_df) == len(validation_df))

Original rows: 541909
Exported CSV rows: 541909
Same row count: True


In [49]:
print(validation_df.columns.tolist())

['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']


## Assessment

Before performing any cleaning, I first assessed the overall quality of the dataset.

* The dataset contains over 541,000 transaction records across 25,900 invoices, involving 4,372 customers from 38 countries
* 135,080 missing Customer IDs
* 5,268 exact duplicate records, requiring validation before removal.
* 10,624 negative quantities and 9,288 cancelled transactions, which are likely to represent product returns or cancelled orders rather than errors.
* 2,515 zero-priced transactions and 1,454 missing product descriptions, requiring further investigation.

### Why don't we clean data immediately?

I want to understand the data itself, rather than modify it. For instance, records such as returns and canelled orders may appear incorrect but actually represent important business events, therefore removing them will take the risk of losing insights. 

I decided to clean and transform later in SQL where each business rule could be documented and reproduced.

### Staging Dataset

A staging dataset was created to prepare the data for SQL. It only includes technical standardisation, such as consistent column names, data types and date formats, while preserving all original business records.

This allows users to compare the staging data with the raw source before any SQL transformations are applied, ensuring a transparent and reproducible data cleaning process.